# SafeFall AI - AI-Powered Elderly Fall Detection System
### Formative Assessment-2 (FA-2: Model Selection, Training, Evaluation & Deployment)
**Course:** Machine Learning and Deep Learning | **Organization:** CareVision HealthTech Pvt. Ltd.

---

## Project Overview & Objectives
Elderly falls are a leading cause of fatal and non-fatal injuries worldwide. In FA-1, we explored patient safety challenges, analyzed the **Le2i Fall Dataset**, and designed our AI storyboard.
In **FA-2**, we build, train, evaluate, and deploy **SafeFall AI**:
1. **Pose Estimation:** Google MediaPipe Pose (33 3D body keypoints).
2. **Feature Engineering:** Biomechanical metrics (Torso Angle $\theta$, Aspect Ratio $W/H$, Knee & Hip flexion, Center of Gravity $y$).
3. **Activity Classification:** Deep Neural Network (PyTorch MLP) + Random Forest baseline across **5 classes**:
   - `Fall Detected` (Emergency Alert Trigger)
   - `Walking`
   - `Sitting`
   - `Standing`
   - `Normal Activity`
4. **Dataset Splitting:** 70% Training, 15% Validation, 15% Testing.
5. **Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, Confusion Matrix, Loss & Accuracy graphs.
6. **Deployment:** Interactive Streamlit Cloud Dashboard with Real-Time Emergency Alerts.


## Step 1: Environment Setup & Library Imports


In [ ]:
import os
import sys
import json
import time
import joblib
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

print("PyTorch Version:", torch.__version__)
print("OpenCV Version:", cv2.__version__)
print("Device Available:", "CUDA GPU" if torch.cuda.is_available() else "CPU")


## Step 4: Model Selection - Choosing the Right AI Tool

### 1. Pose Estimation Comparison:
- **MediaPipe Pose (Selected):** Lightweight (runs at >30 FPS on standard CPUs), requires no heavy GPU runtime, extracts 33 3D normalized body landmarks, highly robust for real-time edge healthcare.
- **YOLOv8-Pose:** High accuracy on multi-person scenes, but computationally heavier for single-patient room monitoring.
- **OpenPose:** High accuracy but slow inference on CPU systems.

### 2. Activity Classifier Architecture:
We design a **Deep Neural Network (SafeFallDeepNet)** in PyTorch utilizing 142 features (132 normalized landmark coordinates + 10 kinematic/biomechanical indicators) and benchmark against a **Random Forest** baseline.


In [ ]:
CLASSES = [
    "Fall Detected",
    "Walking",
    "Sitting",
    "Standing",
    "Normal Activity"
]

class SafeFallDeepNet(nn.Module):
    """
    Deep Neural Network for Human Activity & Fall Detection.
    Takes 142 input features (132 landmark values + 10 biomechanical features).
    """
    def __init__(self, input_dim=142, num_classes=5, dropout_rate=0.3):
        super(SafeFallDeepNet, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.7),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        return self.network(x)

model_summary = SafeFallDeepNet(142, 5)
print(model_summary)


## Step 5: Model Design & Training (70% Train / 15% Val / 15% Test)

We load the preprocessed dataset structured according to the Le2i Fall Benchmark:
- **70% Training Set:** 1,400 samples
- **15% Validation Set:** 300 samples
- **15% Testing Set:** 300 samples


In [ ]:
# Load Datasets
data_dir = "../data"
train_df = pd.read_csv(os.path.join(data_dir, "train_dataset.csv"))
val_df = pd.read_csv(os.path.join(data_dir, "val_dataset.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test_dataset.csv"))

print("Dataset Shapes:")
print(f"Train Shape: {train_df.shape} (70%)")
print(f"Val Shape:   {val_df.shape} (15%)")
print(f"Test Shape:  {test_df.shape} (15%)")

feature_cols = [c for c in train_df.columns if c != "activity_label"]

# Fit Scaler & Label Encoder
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values.astype(np.float32))
X_val = scaler.transform(val_df[feature_cols].values.astype(np.float32))
X_test = scaler.transform(test_df[feature_cols].values.astype(np.float32))

label_encoder = LabelEncoder()
label_encoder.fit(CLASSES)
y_train = label_encoder.transform(train_df["activity_label"].values)
y_val = label_encoder.transform(val_df["activity_label"].values)
y_test = label_encoder.transform(test_df["activity_label"].values)

print("\nClass Mapping:", dict(zip(label_encoder.classes_, range(len(CLASSES)))))


In [ ]:
# Training PyTorch Neural Network with AdamW & Early Checkpointing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SafeFallDeepNet(input_dim=len(feature_cols), num_classes=5).to(device)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 40
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * bx.size(0)
        _, preds = torch.max(out, 1)
        correct += (preds == by).sum().item()
        total += by.size(0)
    
    t_loss = running_loss / total
    t_acc = (correct / total) * 100.0
    
    model.eval()
    val_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            out = model(bx)
            loss = criterion(out, by)
            val_loss += loss.item() * bx.size(0)
            _, preds = torch.max(out, 1)
            v_corr += (preds == by).sum().item()
            v_tot += by.size(0)
            
    v_loss = val_loss / v_tot
    v_acc = (v_corr / v_tot) * 100.0
    
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accs.append(t_acc)
    val_accs.append(v_acc)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] Train Loss: {t_loss:.4f}, Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc:.2f}%")


## Step 6: Model Evaluation & Testing - How Good Is Your Model?

We test the trained model on the **held-out Test Set (300 samples)** that the model has never encountered.
We compute:
- **Accuracy:** Overall correctness across all 5 classes.
- **Precision:** Exactness of fall detection (minimizing false alarms).
- **Recall:** Sensitivity to real falls (crucial in healthcare to never miss a fall event).
- **F1-Score:** Harmonic mean of precision and recall.
- **Confusion Matrix:** Error distribution analysis.


In [ ]:
# Evaluate on Unseen Test Set
model.eval()
with torch.no_grad():
    test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_logits = model(test_tensor)
    test_probs = torch.softmax(test_logits, dim=1).cpu().numpy()
    y_pred = np.argmax(test_probs, axis=1)

print("=== SafeFall AI: Test Set Classification Report ===")
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6), dpi=150)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("SafeFall AI - Test Set Confusion Matrix", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Plot Training & Validation Curves
plt.figure(figsize=(14, 5), dpi=150)

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_accs, label="Train Accuracy", color="#1E88E5", lw=2)
plt.plot(range(1, epochs + 1), val_accs, label="Val Accuracy", color="#43A047", lw=2, linestyle="--")
plt.title("SafeFall AI - Accuracy Curve", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)

# Loss
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), train_losses, label="Train Loss", color="#E53935", lw=2)
plt.plot(range(1, epochs + 1), val_losses, label="Val Loss", color="#FB8C00", lw=2, linestyle="--")
plt.title("SafeFall AI - Cross-Entropy Loss Curve", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


## Real-World Deployment Challenges & Mitigation Strategies

1. **Lighting Variations (Day vs. Night):**
   - *Challenge:* Low-light surveillance video in bedrooms causes landmark tracking degradation.
   - *Mitigation:* MediaPipe operates on normalized gradients; adaptive histogram equalization (CLAHE) is applied to enhance contrast prior to landmark extraction.
2. **Camera Angles & Perspective Distortion:**
   - *Challenge:* Overhead vs wall-mounted angles distort apparent height.
   - *Mitigation:* We extract rotation-invariant 3D keypoint vectors and joint angle ratios rather than absolute pixel distances.
3. **Occlusion (Beds, Chairs, Tables):**
   - *Challenge:* Lower body keypoints occluded by furniture.
   - *Mitigation:* MediaPipe predicts landmark visibility scores; upper-body kinematic fallback and heuristic center-of-gravity drop detection compensate for missing lower limbs.
4. **Similar Body Postures (Lying on couch vs. falling):**
   - *Challenge:* Person intentionally resting horizontally on a bed or sofa.
   - *Mitigation:* Temporal confirmation window (confirming fallen state persists on the floor elevation for >3 seconds) and room floor plane bounding prevent false alerts.


## Step 7 & 8: Streamlit Cloud Deployment & Continuous Monitoring

### Streamlit Healthcare Dashboard Structure (`app.py`):
- **Image Diagnostic Hub:** Upload patient snapshots, visualize 33 MediaPipe pose landmarks, inspect joint angles, and run real-time classification.
- **Video Stream Analyzer:** Upload surveillance `.mp4`/`.avi` clips, generate frame-by-frame activity timeline, and log fall incident timestamps.
- **Live Webcam Mode:** Real-time patient monitoring with heads-up display (HUD), posture angle gauge, and emergency buzzer alert.
- **Performance Analytics Center:** Interactive Confusion Matrix, Accuracy & Loss telemetry, and class distribution breakdown.
- **Emergency Dispatch Panel:** Automated SMS dispatch simulation, caregiver contact directory, and downloadable incident logs.

### Deployment Instructions:
```bash
pip install -r requirements.txt
streamlit run app.py
```
